### Necessary Steps for pre-processing data

- Article Type
    - discard all except 'research-article', 'review-article', 'case-report'
- Sections
    - determine which sections are necessary for inclusion (maybe determine different criteria for research vs. reviews)
    - create dict that maps each section to its standardized section name
    - discard all other papers and superfluous sections
    - clean remaining sections (remove citations?) (do this after count vectorizing when outliers become apparent)
- Date
    - reshape date format
    - sort by date 
    - re-structure the data frames, such that each data frame contains all papers from a given year
- Language
    - infer langugage for all papers without specified language
    - discard all non-english



In [ ]:
import pandas as pd
from pandas._libs.tslibs.timestamps import Timestamp
import re
import os
from collections import Counter
from polyglot.detect import Detector
from polyglot.detect.base import Error

%load_ext autoreload
%autoreload 2
import src.data_utils as utils

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
BASELINE_NAME = "baseline_2025-06-26"

In [ ]:
baseline_df = pd.read_json("../data/" + BASELINE_NAME + "/PMC008xxxxxx_noncomm.json")

In [ ]:
baseline_df.head()

In [ ]:
paper_count_0 = len(baseline_df)
paper_count_0

#### Article Type

In [ ]:
include_types = ["research-article", "review-article", "case-report"]
baseline_df = baseline_df[
    baseline_df["article-type"].apply(lambda x: x in include_types)
]
len(baseline_df)

#### Sections

In [ ]:
# replace None with empty string to make my life easier
baseline_df["section_titles"] = baseline_df["section_titles"].apply(
    lambda x: [] if x == None else x
)

In [ ]:
test_secs = [
    ["this is an intro.", "these are methods.", "this is a result."],
    [
        "bla bla bla",
        "this is the first method",
        "bla bla bla bla",
        "this is the second method",
    ],
    ["this is being ignored"],
]

test_titles = [
    ["introduction", "method", "result"],
    ["bla", "method", "bla", "method"],
    [],
]

utils.standardize_sections(test_secs, test_titles)

In [ ]:
baseline_df["sections"] = utils.standardize_sections(
    list(baseline_df["sections"]), list(baseline_df["section_titles"])
)
baseline_df.head()

In [ ]:
# remove papers with empty sections dictionaries
baseline_df = baseline_df[baseline_df["sections"].apply(lambda x: not x == {})]
len(baseline_df)

#### Language

In [ ]:
Detector(baseline_df["abstract"].iloc[0]).languages[0].code

In [ ]:
Detector("abstracts are an important part of papers").languages[0].code

In [ ]:
baseline_df["language"] = utils.determine_lang(
    baseline_df["language"], list(baseline_df["abstract"])
)

In [ ]:
# remove non-english entries
baseline_df = baseline_df[baseline_df["language"].apply(lambda x: x == "en")]
len(baseline_df)

#### Dates

In [ ]:
baseline_df["date"] = utils.get_alt_date(baseline_df["date"])

In [ ]:
baseline_df = baseline_df.sort_values("date", ignore_index=True)